In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

**<font size="6" color="red">ch2. Ollama_LLM활용의 기본 개념(LangChain)</font>**

# 1.  LLM 을 활용하여 답변 생성
## 1) Ollama 이용한 로컬 LLM 이용
- 성능은 GPT(open ai API), Claude같은 모델보다 떨어지나, 개념 설명을 위해 open source 모델 사용

### ollama.com 설치 -> 모델 pull
- cmd창에서 ollama pull deepseek-r1:1.5b

In [2]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='deepseek-r1:1.5b')
result = llm.invoke('What is the capital of France?')
result # AIMessage
# content : 실제 답변
# response_metadata : 모델 실행에 대한 상세 정보(전체소요시간, 모델로딩시간, 처리토큰수)

AIMessage(content='\n\nThe capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2026-09-11T07:07:21.8845227Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2012880100, 'load_duration': 1892826400, 'prompt_eval_count': 10, 'prompt_eval_duration': 63639000, 'eval_count': 12, 'eval_duration': 47347000, 'logprobs': None, 'model_name': 'deepseek-r1:1.5b', 'model_provider': 'ollama'}, id='lc_run--01a08f4a-cfbd-7403-b184-f203f66a5dcc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 10, 'output_tokens': 12, 'total_tokens': 22})

In [3]:
print(result.content)



The capital of France is Paris.


### 모델 pull
- ollama pull llama3.2:1b
- ollama 모델은 공식적으로 한글지원 안 됨(llama3.1:405b 한글지원 가능 -> llama3.2:3b한글 지원이 일부)
- exaone 모델은 공식적으로 한글지원 : ollama pull exaone3.5:2.4b

- 모델 저장 경로 : C:\Users\내컴퓨터이름\.ollama

In [4]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b', 
                temperature=0,        # 결정적인 출력
                num_predict=512,      # 최대 512 토큰까지 생성
                num_ctx=4096,          # 컨텍스트 윈도우 넉넉하게
                top_k=40,
                top_p=0.9,
                stop=["\n\n"])          # 두 줄바꿈 나오면 생성 중단)
result = llm.invoke('What is the capital of Korea?')
result

AIMessage(content='The capital of South Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-09-11T07:07:23.4033877Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1450605700, 'load_duration': 1366617300, 'prompt_eval_count': 32, 'prompt_eval_duration': 37848000, 'eval_count': 9, 'eval_duration': 41578000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--01a08f4a-d7de-74a2-a479-cad1f5018cb0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 9, 'total_tokens': 41})

In [5]:
result.content

'The capital of South Korea is Seoul.'

In [6]:
llm = ChatOllama(model='exaone3.5:2.4b')
result = llm.invoke('한국 수도는 어디예요?')
result.content

ResponseError: model 'exaone3.5:2.4b' not found (status code: 404)

## 2) openai 모델 활용
- pip install langchain-openai

In [ ]:
# 환경변수(`OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable)
from dotenv import load_dotenv
import os
load_dotenv()
# print(os.getenv('OPENAI_API_KEY'))
# print(os.environ['OPENAI_API_KEY'])

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano",
               # api_key=os.getenv('OPENAI_API_KEY')
                )
#result = llm.invoke('What is the capital of Korea?')
result = llm.invoke('한국의 수도가 어디예요?')
result.content

In [ ]:
result

In [ ]:
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model='claude-haiku-4-5-20251001')
# ANTHROPIC_API_KEY environment variable
# llm.invoke('What is the capital of Korea?')

# 2. 렝체인 스타일로 프롬프트 작성하기
- 프롬프트 : llm호출시 쓰는 질문

## 1) 기본 프롬프트 템플릿 사용
- PromptTemplate 을 사용하여 변수가 포함된 템플릿 작성하면 PromptValue를 만들 수 있다

In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
# llm.invoke(0)
llm.invoke("What is the capital of Korea")
# 프롬프트 가능 타입 : str, PromptValue, list of BaseMessages

In [ ]:
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate(
                template="What is the capital of {country}?", # {}안에 값을 새로운 값으로 대체
                input_variables = ['country']
        )
prompt = prompt_template.invoke({"country":"Korea"})
print(1, type(prompt))
prompt = prompt_template.invoke("Korea")
print(2, prompt)
llm.invoke(prompt)

In [ ]:
country = input('수도를 알고 싶은 나라는(영어)?')
llm.invoke(prompt_template.invoke(country))

In [ ]:
def answer(country):
    '나라명을 입력받아 llm에게 수도명을 받아 return'
    from langchain_ollama import ChatOllama
    from langchain_core.prompts import PromptTemplate
    llm = ChatOllama(model='llama3.2:1b')
    prompt_template = PromptTemplate(
                        template='What is the capital of {country}?',
                        input_variables = ['country']
                    )
    result = llm.invoke(prompt_template.invoke(country))
    return result.content

In [ ]:
country = input("수도를 알고 싶은 나라는(영어)>")
answer(country)

## 2) 메세지 기반 프롬프트 작성
- list of BaseMessages
- BaseMessage 상속받은 클래스 : AIMessage, HumanMessage, SystemMessage, ToolMessage
- [BaseMessage객체, BaseMessage객체, BaseMessage객체, ...]

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
message_list = [
    SystemMessage(content="You are a helpful assistant!"), # llm 페로소나
    HumanMessage(content="What is the capital of Italy?"), # 질문 답변 예제(few shot)
    AIMessage(content='The capital of Italy is Rome.'),
    HumanMessage(content="What is the capital of France?"), # 질문 답변 예제(few shot)
    AIMessage(content='The capital of France is Paris.'),
    HumanMessage(content='What is the capital of Korea?') # llm에게 질문하고 싶은 진짜 내용
]
llm.invoke(message_list)

## 3) ChatPromptTemplate 사용(추천; 확장성 용이)
- BaseMessage 리스트 -> 튜플리스트

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate([
    SystemMessage(content="You are a helpful assistant!"), # llm 페로소나
    HumanMessage(content="What is the capital of Italy?"), # 질문 답변 예제(few shot)
    AIMessage(content='The capital of Italy is Rome.'),
    HumanMessage(content="What is the capital of France?"), # 질문 답변 예제(few shot)
    AIMessage(content='The capital of France is Paris.'),
    HumanMessage(content='What is the capital of {country}?') # llm에게 질문하고 싶은 진짜 내용
])
prompt = chatPromptTemplate.invoke({'country':'Korea'})
print('프롬프트 :', prompt)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate([
    ("system", "You are a helpful assistant!"), # llm 페로소나    
    ("human", "What is the capital of Italy?"), # few shot
    ("ai", "The capital of Italy is Rome."),
    ("human", "What is the capital of France?"),
    ("ai", "The capital of France is Paris."),
    ("human", "What is the capital of {country}?")
])
prompt = chatPromptTemplate.invoke({'country':'Korea'})
print('프롬프트 :', prompt)

In [ ]:
# llm.invoke(chatPromptTemplate.invoke({'country':'Korea'}))
llm.invoke(chatPromptTemplate.invoke({'Korea'}))

# 3. 답변 형식 컨트롤하기
- invoke 실행 결과 AIMessage() -> String, json 변환해주는 OutputParser 이용

## 1) 문자열 출력 파서 이용
- StrOutputParser를 이용하여 LLM출력(AIMessage)를 단순 문자열로 변환

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(
    template='What is the capital of {country}? Return the name of the city only.',
    input_variables = ['country']
)
# 프롬프트 템플릿에 값 주입
prompt = prompt_template.invoke({'country':'Korea'})
print('프롬프트 :', prompt)
# llm에 질문
aimessage = llm.invoke(prompt)
# aimessage중 답변만 문자로 받기
output_parser = StrOutputParser()
result = output_parser.invoke(aimessage)
print('파서 결과 :',result)

In [ ]:
output_parser.invoke(llm.invoke(prompt_template.invoke({'country':'Korea'})))
output_parser.invoke(llm.invoke(prompt_template.invoke('Korea')))

In [ ]:
chatPromptTemplate = ChatPromptTemplate([
    ("system", "You are a helpful assistant!"), # llm 페로소나    
    ("human", "What is the capital of Italy?"), # few shot
    ("ai", "The capital of Italy is Rome."),
    ("human", "What is the capital of France?"),
    ("ai", "The capital of France is Paris."),
    ("human", "What is the capital of {country}? Return the name of the city only.")
])
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(chatPromptTemplate.invoke({'country':'Korea'})))

## 2) Json 출력파서 이용
- {'name':'홍','age':20}

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
llm = ChatOllama(model='llama3.2:1b')
country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}.
        - Capital
        - Population
        - Language
        - Currency
    Return ONLY a valid JSON object with no additional text.
    Example format :
    {{"Capital":"Seoul", "Population":"50 million", "Language":"Korean", "Currency":"won"}}
    """,
    input_variables = ['country']
)
prompt = country_detail_prompt.invoke({'country':'France'})
aimessage = llm.invoke(prompt)
output_parser = JsonOutputParser()
result = output_parser.invoke(aimessage)
print(type(result), result)

In [ ]:
info = output_parser.invoke(llm.invoke(country_detail_prompt.invoke('France')))
info

## 3) 구조화된 객체로 반환
- Pydantic 모델(pip show pydantic)을 사용하여 LLM출력을 구조화된 형식으로 받기(JsonParser 좀 안정적)
- Pydantic : 데이터 유효성 검사, 설정관리를 간편하게 해주는 라이브러리

In [ ]:
class User:
    def __init__(self, id, name, is_active=True):
        self.id = id
        self.name = name
        self.is_active = is_active
user = User('1', "홍길동")
user = User(1, "홍", "활성화됨")
print(user)
print(user.id, user.name, user.is_active)

In [ ]:
from pydantic import BaseModel, Field
class User(BaseModel):
    # gt=0 : id>0, lt=0 : id<0, ge=0 : id>=0, le=0: id<=0
    id:int         = Field(gt=0,         description='id')
    name:str       = Field(min_length=2, description='name')
    is_active:bool = Field(default=True, description='id활성화')
user = User(id="1", name='홍길동', is_active=True)
print(user)

In [ ]:
country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}.
        - Capital
        - Population
        - Language
        - Currency
    Return ONLY a valid JSON object with no additional text.
    """,
    input_variables = ['country']
)
class CountryDetail(BaseModel):
    capital:str   = Field(description="the capital of the country")
    population:int= Field(description="the population of the country")
    language:str  = Field(description="the language of the country")
    currency:str  = Field(description="the currency of the country")
# 출력파서 + llm
structedllm = llm.with_structured_output(CountryDetail)
info = structedllm.invoke(country_detail_prompt.invoke({'country':'Korea'}))
print(type(info))
print(info)
print(info.capital, info.population, info.language, info.currency)
print(info.model_dump()) # 객체를 dict로

In [ ]:
aimessage = llm.invoke(country_detail_prompt.invoke({'country':'Korea'}))
print(aimessage.content)

# 4. LCEL(LangChain Expression Language)을 활용한 렝체인 생성
## 1) 문자열 출력 파서 사용
- StrOutputParser, ChatOllama, PromptTemplate등은 모두 Runable로 상속받아 invoke가 있음

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(
    template='What is the capital of {country}? Return the name of the city only.',
    input_variables = ['country']
)
outputParser = StrOutputParser()
outputParser.invoke(llm.invoke(prompt_template.invoke({'country':'Korea'})))

## 2) LCEL을 사용한 체인 구성
- (|) 사용

In [ ]:
# 프롬프트 템플릿 -> llm -> 출력파서를 연결시키는 체인 생성
capital_chain = prompt_template | llm | outputParser
# 생성된 체인 invoke
capital_chain.invoke({'country':'Korea'})
capital_chain.invoke('Korea')

In [ ]:
type(capital_chain)

## 3) 복합 체인 구성
- 여러 단계의 추론이 필요한 경우 (체인 연결)

In [ ]:
# 나라 설명 -> 나라이름
country_prompt = PromptTemplate(
                    template="""
                    Guess the name of the country based on the following information:
                    {information}
                    Return the name of the country only""",
                input_variables=['information']
)
outputParser.invoke(llm.invoke(country_prompt.invoke({'information':
                                            "This country is very famous for its wine"})))

In [ ]:
# 나라명 추측 체인
country_chain = country_prompt | llm | outputParser
country_chain.invoke({'information':'This country is very famous for its wine'})

In [ ]:
# 국가설명 (-> 국가명 ->) 그 국가 수도명
final_chain = country_chain | capital_chain
k.invoke({'information':'This country is very famous for its wine'})

In [ ]:
# 국가설명 -> 국가명 -> 그 국가 수도명
final_chain = {'country':country_chain} | capital_chain

final_chain.invoke({'information':'This country is very famous for its wine'})

In [ ]:
final_chain.invoke({'information':'와인으로 유명해요'})

```
LLM호출 : ChatOllama(llama3.2:1b / exaone3.5:2.4b), ChatOpenAI
LLM 호출에 필요한 프롬프트 템플릿 : PromptTemplate, ChatPromptTemplate(few shot, 페르소나설정)
LLM 결과를 변환 outputParser : str/JSON/pydentic 클래스 생성하여 LLM의 Structed outputParser 이용
위 모두 runnable로부터 상속받은 invoke 가능 => LangChain으로 연결 가능 (=>RAG)
```
# 5. 생성형 AI  평가 : 
- 첫번째 체인 : 나라이름 -> 그 나라에서 가장 유명한 음식
- 두번째 체인 : 음식 -> 음식의 레시피
- 최종 체인 : 나라이름 -> 그 나라에 가장 유명한 음식의 레시피